# Módulo 4: Aprendizaje Automático (Machine Learning)
## Reto 06 - Fase 2: Modelado y Comparación de Clasificadores (versión del estudiante)

En la Fase 1 (`06_Exploracion_Datos.ipynb`) conocimos los datos y terminamos con **hipótesis**. Ahora toca comprobarlas con evidencia.

**Hipótesis de la Fase 1:**
| Modelo | Predicción |
|---|---|
| **k-NN** | Bueno, siempre que se escale. Los valores extremos y la zona gris lo pueden confundir. |
| **Árbol de Decisión** | Favorito: hay umbrales claros y captura la "U" de la hora. Riesgo de *overfitting* sin límite de profundidad. |
| **Naive Bayes** | Detectará los ataques fáciles, con más falsas alarmas en la zona gris. El logaritmo debería ayudarle. |

Y una pregunta abierta: **¿qué pasa si quitamos `login_exitoso`**, la columna que solo se conoce al final de la sesión?

**Cómo trabajar este notebook:**
* Cada paso explica **qué** hacer. Las celdas de código están vacías: el código lo escribe usted.
* Después de cada resultado hay una celda ** Tu respuesta** para interpretar lo que obtuvo.
* Use `random_state=42` en todo lo que tenga azar, para que sus resultados se puedan comparar con los de la clase.

**Pasos:**
1. Preparar los datos
2. Separar *train* y *test*
3. Construir los tres modelos con *pipelines*
4. Escenario A: entrenar y evaluar **con** `login_exitoso`
5. Validación cruzada
6. Poner a prueba cada hipótesis
7. Escenario B: **sin** `login_exitoso` (detección en tiempo real)
8. Interpretar el Árbol de Decisión
9. Decisión final
10. ¿Se cumplieron las hipótesis?

---
## **Paso 1: Preparación de los datos**

**Instrucciones:**
1. Importe las librerías que va a necesitar:
   * `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`, `time`
   * De `sklearn.model_selection`: `train_test_split`, `StratifiedKFold`, `cross_validate`
   * De `sklearn.pipeline`: `make_pipeline`
   * De `sklearn.compose`: `ColumnTransformer`
   * De `sklearn.preprocessing`: `StandardScaler`, `FunctionTransformer`
   * De `sklearn.impute`: `SimpleImputer`
   * Los tres modelos: `KNeighborsClassifier`, `DecisionTreeClassifier` (y `plot_tree`, `export_text`), `GaussianNB`
   * De `sklearn.metrics`: `accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `confusion_matrix`, `ConfusionMatrixDisplay`
2. Ejecute `sklearn.set_config(transform_output='pandas')` para que los pasos del pipeline devuelvan DataFrames con nombres de columna. Lo necesitará en el Paso 6.
3. Cargue `06_dataset_logins.csv`.
4. Aplique las decisiones de limpieza de la Fase 1: elimine `id_sesion` y las filas duplicadas.
5. Imprima cuántas sesiones quedan. *(Debería obtener 3000.)*

**Instrucciones:**
1. Cree dos listas de predictores:
   * `PREDICTORES_CON_LOGIN`: las 7 columnas predictoras.
   * `PREDICTORES_SIN_LOGIN`: las mismas, sin `login_exitoso`.
2. Cree la lista `CONTINUAS` con las columnas sesgadas: `intentos_fallidos`, `intentos_por_minuto`, `usuarios_distintos`, `duracion_sesion_seg`.
3. Guarde la variable objetivo `es_ataque` en `y`.
4. Cree una máscara booleana `zona_gris` con la definición de la Fase 1: menos de 4 intentos por minuto, 3 o más intentos fallidos y 2 usuarios distintos o menos.

---
## **Paso 2: Separación de datos (Train/Test Split)**

Separaremos 70% para aprender y 30% para el examen.

**Instrucciones:**
1. Use `train_test_split` sobre los **índices** del DataFrame (`df.index`), no sobre las columnas. Así los escenarios A y B usarán **exactamente las mismas sesiones** de examen.
   * `test_size=0.30`, `random_state=42`, `stratify=y`
2. Obtenga `y_train` y `y_test` con `.loc`.
3. Cree `zona_test`: la máscara de zona gris aplicada solo a las filas del examen.
4. Muestre una tabla con, para *train* y para *test*: cantidad de sesiones, cantidad de ataques y % de ataques.
5. Imprima cuántas sesiones de la zona gris quedaron en el examen.

**Pregunta:** ¿Por qué es importante usar `stratify=y` en este dataset? ¿Qué pasaría con el % de ataques si no lo usáramos?

 **Tu respuesta:**

---
## **Paso 3: Construcción de los modelos con *pipelines***

Un **pipeline** encadena la preparación y el modelo en un solo objeto. Cuando se llama a `.fit()`, todo (la mediana para rellenar nulos, la media y desviación del escalado) se calcula **solo con los datos de entrenamiento**. Así se evita la fuga de información (*data leakage*).

| Modelo | Pasos del pipeline |
|---|---|
| k-NN | `SimpleImputer(strategy='median')` → `StandardScaler()` → `KNeighborsClassifier(n_neighbors=5)` |
| Árbol de Decisión | `SimpleImputer(strategy='median')` → `DecisionTreeClassifier(max_depth=4, random_state=42)` |
| Naive Bayes | `SimpleImputer(strategy='median')` → `GaussianNB()` |

**Instrucciones:**
1. Escriba una función `crear_modelos()` que devuelva un diccionario `{nombre: pipeline}` con los tres modelos de la tabla. *(Usar una función permite crear modelos "nuevos" sin entrenar cada vez que se necesiten.)*
2. Escriba una función `evaluar(modelo, X_tr, y_tr, X_te, y_te)` que:
   * mida el tiempo con `time.perf_counter()`,
   * entrene el modelo y prediga el examen,
   * devuelva las predicciones y un diccionario con: Accuracy, Precision, Recall, F1, Falsos negativos, Falsos positivos y Tiempo (ms).

**Pregunta:** ¿Por qué k-NN lleva `StandardScaler` en su pipeline y el Árbol y Naive Bayes no?

 **Tu respuesta:**

### Recordatorio: ¿qué mide cada métrica?
Piense en el modelo como una alarma del SOC:
* **Precision**: de todas las alertas que lanzó, ¿cuántas eran ataques reales? *(baja precision = muchas falsas alarmas)*
* **Recall**: de todos los ataques reales, ¿cuántos detectó? *(bajo recall = ataques que pasan sin ser vistos)*
* **F1**: el balance entre ambas (media armónica).
* **Accuracy**: % total de aciertos. En datos desbalanceados es la menos útil.

---
## **Paso 4: Escenario A - Con `login_exitoso` (análisis posterior)**

Este escenario representa una revisión **forense**: el analista revisa las sesiones cuando ya terminaron y conoce su resultado final.

**Instrucciones:**
1. Cree `X_train_A` y `X_test_A` con `PREDICTORES_CON_LOGIN` y los índices del Paso 2.
2. Recorra los modelos de `crear_modelos()`, evalúelos con `evaluar()` y guarde:
   * los modelos entrenados en `modelos_A`,
   * las predicciones en `predicciones_A`,
   * las métricas en un DataFrame `resultados_A`.
3. Muestre `resultados_A` redondeado a 3 decimales.

**Instrucciones:** dibuje las tres **matrices de confusión** una al lado de la otra (`plt.subplots(1, 3)` y `ConfusionMatrixDisplay`). Use las etiquetas `['Normal', 'Ataque']` y ponga el nombre del modelo como título de cada una.

**Preguntas:**
1. ¿Los tres modelos superan al modelo "perezoso" que siempre dice "Normal" (77.8%)?
2. Si solo mirara la *accuracy*, ¿diría que hay mucha diferencia entre los modelos?
3. Según las matrices: ¿qué modelo deja escapar **menos ataques**? ¿Cuál lanza **menos falsas alarmas**? ¿Es el mismo modelo?
4. En seguridad, ¿qué casilla de la matriz es la más peligrosa y por qué?

 **Tu respuesta:**

---
## **Paso 5: Validación cruzada**

Un solo examen puede tener suerte. La validación cruzada divide los datos de entrenamiento en 5 partes y entrena 5 veces, cada vez evaluando con una parte distinta. La **media** dice qué tan bueno es el modelo y la **desviación estándar** qué tan estable es.

**Instrucciones:**
1. Cree `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
2. Para cada modelo, use `cross_validate` sobre `X_train_A` y `y_train` (¡no use el *test*!) con `scoring=['recall', 'precision', 'f1']`.
3. Muestre una tabla con la media ± desviación de cada métrica por modelo.
4. Guarde las 5 notas de cada fold en una lista para graficarlas.

**Instrucciones:** dibuje un `sns.boxplot` con las notas por fold: eje X = métrica, eje Y = valor, color (`hue`) = modelo.

**Preguntas:**
1. ¿La validación cruzada confirma lo que vio en el examen del Paso 4?
2. ¿Las desviaciones son grandes o pequeñas? ¿Qué dice eso sobre la estabilidad de los modelos?

 **Tu respuesta:**

---
## **Paso 6: Poniendo a prueba las hipótesis**

### 6.1 Hipótesis k-NN: "necesita escalado"

**Instrucciones:**
1. Escriba una función `transformador_log(columnas)` que devuelva un `ColumnTransformer` que:
   * aplique `FunctionTransformer(np.log1p, feature_names_out='one-to-one')` solo a las columnas de `CONTINUAS` que estén en `columnas`,
   * deje pasar las demás columnas sin cambios (`remainder='passthrough'`, `verbose_feature_names_out=False`).
2. Compare tres versiones de k-NN (k=5) con `evaluar()`:
   * sin escalar,
   * escalado (el modelo base),
   * `log1p` + escalado.
3. Muestre Precision, Recall, F1, Falsos negativos y Falsos positivos de cada versión.

**Pregunta:** ¿Se confirma la hipótesis? ¿Qué versión es la mejor y por qué cree que el logaritmo ayuda a k-NN?

 **Tu respuesta:**

### 6.2 Hipótesis Árbol: "sin límite de profundidad memoriza los datos"

**Instrucciones:**
1. Para cada profundidad en `[1, 2, 3, 4, 5, 6, 8, 10, 15, None]`, cree un árbol y use `cross_validate` con `scoring='f1'` y `return_train_score=True`.
2. Grafique dos líneas: F1 en **entrenamiento** y F1 en **validación**, en función de la profundidad.

**Instrucciones:** compare el árbol `max_depth=4` contra un árbol **sin límite**. Para cada uno muestre:
* accuracy en *train*,
* Precision, Recall, F1 y Falsos positivos en *test*,
* precision dentro de la zona gris,
* número de hojas (`get_n_leaves()`) y profundidad real (`get_depth()`).

*Pista:* el árbol es el último paso del pipeline: `pipeline[-1]`.

**Preguntas:**
1. ¿A partir de qué profundidad deja de mejorar la validación?
2. ¿Qué pasa con la nota de entrenamiento a medida que aumenta la profundidad? ¿Cómo se llama ese fenómeno?
3. Si los dos árboles tienen un F1 parecido en el examen, ¿cuál preferiría y por qué?

 **Tu respuesta:**

### 6.3 Hipótesis Naive Bayes: "los supuestos rotos le afectan; el logaritmo le ayudará"

**Instrucciones:**
1. Compare Naive Bayes **base** contra Naive Bayes con `transformador_log` antes del modelo.
2. Muestre Precision, Recall, F1, Falsos negativos, Falsos positivos, y recall y precision **dentro de la zona gris**.

**Instrucciones:** ¿Naive Bayes es demasiado confiado? Para cada modelo del Escenario A:
1. Obtenga las probabilidades con `predict_proba(X_test_A)[:, 1]`.
2. Calcule el % de predicciones con certeza mayor a 99%. *(La certeza es `max(p, 1 - p)`.)*
3. Calcule la **certeza media cuando el modelo se equivoca**.

**Preguntas:**
1. ¿El logaritmo ayudó a Naive Bayes? ¿La hipótesis se confirma o se rechaza?
2. El logaritmo arregla las distribuciones sesgadas. ¿Qué **otro** supuesto de Naive Bayes sigue roto? *(Recuerde la matriz de correlación de la Fase 1.)*
3. ¿Por qué es peligroso para un analista del SOC un modelo que se equivoca con 98% de certeza?

 **Tu respuesta:**

### 6.4 Hipótesis zona gris: "aquí se decide qué modelo es mejor"

**Instrucciones:**
1. Para cada modelo del Escenario A, calcule recall y precision **globales** y **dentro de la zona gris** (use `zona_test`).
2. Grafique en barras el recall global contra el recall en la zona gris.

**Preguntas:**
1. ¿Qué le pasa al recall de todos los modelos dentro de la zona gris?
2. ¿Qué modelo encuentra más ataques en la zona gris? ¿A qué precio?
3. ¿Qué información adicional (que no está en el dataset) podría ayudar a resolver la zona gris?

 **Tu respuesta:**

---
## **Paso 7: Escenario B - Sin `login_exitoso` (detección en tiempo real)**

En tiempo real no sabemos si el atacante acertará. Repita todo **sin** esa columna, con las **mismas** sesiones de entrenamiento y examen.

**Instrucciones:**
1. Cree `X_train_B` y `X_test_B` con `PREDICTORES_SIN_LOGIN`.
2. Entrene y evalúe los tres modelos igual que en el Paso 4. Guarde `modelos_B`, `predicciones_B` y `resultados_B`.
3. Muestre una tabla que compare los escenarios A y B lado a lado (*pista:* `pd.concat({...}, axis=1)`).

**Instrucciones:** grafique Precision, Recall y F1 de cada modelo en ambos escenarios (*pista:* `sns.catplot(kind='bar', col='Métrica', hue='Escenario')`).

**Preguntas:**
1. `login_exitoso` era la columna más correlacionada con el objetivo. ¿Cuánto cambian los modelos al quitarla?
2. ¿Por qué cree que pasa eso? *(Pista: ¿qué otras columnas cuentan la misma historia?)*

 **Tu respuesta:**

### Los ataques más peligrosos: los que SÍ lograron entrar

**Instrucciones:**
1. Cree una máscara con los ataques del examen que tuvieron `login_exitoso == 1`. Imprima cuántos son.
2. Para cada modelo, cuente cuántos de esos ataques detectó en el Escenario A y cuántos en el B.

**Pregunta:** ¿En qué escenario se detectan mejor los ataques exitosos? ¿Por qué usar `login_exitoso` hace que el modelo pase por alto justamente esos ataques?

 **Tu respuesta:**

---
## **Paso 8: Interpretación del Árbol de Decisión**

Use el árbol del **Escenario B** (el que serviría en tiempo real).

**Instrucciones:**
1. Dibuje el árbol con `plot_tree` (use `feature_names`, `class_names=['Normal', 'Ataque']`, `filled=True`).
2. Imprima sus reglas en texto con `export_text`.
3. Grafique la importancia de cada columna (`feature_importances_`) en barras horizontales.

**Preguntas:**
1. ¿Cuál es la primera pregunta del árbol? ¿Qué tipos de ataque atrapa?
2. Escriba en palabras dos reglas del árbol, como se las explicaría a un analista del SOC.
3. ¿Qué columnas **no** usa el árbol? ¿Por qué cree que las ignora?
4. ¿La hora del día aparece? ¿Coincide con lo que vio en la Fase 1?

 **Tu respuesta:**

---
## **Paso 9: Decisión final**

**Instrucciones:**
1. Ejecute la validación cruzada del Escenario B (recall y F1) para los tres modelos.
2. Construya una tabla final con, por modelo: Precision, Recall, F1, Falsos negativos, Falsos positivos, Tiempo (ms), Recall CV, F1 CV, desviación de F1 CV y si el modelo **explica sus decisiones**.

**Pregunta:** complete la tabla y escriba su recomendación final.

| Si el SOC necesita... | Elegiría | Por qué |
|---|---|---|
| Explicar cada alerta (auditoría, bloqueo de cuentas) | | |
| No dejar escapar casi nada (primer filtro) | | |
| Máxima calidad sin necesidad de explicar | | |

**Mi recomendación final (modelo, configuración y escenario):**

 **Tu respuesta:**

---
## **Paso 10: ¿Se cumplieron las hipótesis?**

Complete la tabla con ✅ (confirmada), ❌ (rechazada) o ⚠️ (parcial), y la evidencia que lo demuestra.

| Hipótesis de la Fase 1 | Resultado | Evidencia (paso y número) |
|---|---|---|
| k-NN necesita escalado | | |
| El Árbol es el favorito | | |
| El Árbol sin límite sobreajusta | | |
| Naive Bayes sufre por sus supuestos | | |
| El logaritmo ayudará a Naive Bayes | | |
| La zona gris decide al ganador | | |
| `login_exitoso` es imprescindible | | |

### Reflexión final
1. ¿La exploración de la Fase 1 le ayudó a anticipar los resultados?
2. ¿Alguna hipótesis resultó falsa? ¿Qué aprendió de eso?
3. ¿Por qué la *accuracy* no era suficiente para elegir el modelo?

 **Tu respuesta:**